# Operator actions and Symbolica export

This notebook walks through the **compiled Lagrangian** layer:

- **`Model` → `model.lagrangian()`** gives a `CompiledLagrangian` of ordered `InteractionTerm`s (the structure the vertex code trusts).
- **`L.to_symbolica()`** turns that into a single Symbolica expression for **display / scalar algebra** (factor order for fermions is **not** preserved in Symbolica).
- **`L.apply_operator(...)`** applies a graded Leibniz rule at the term level; optional **`flavor_expand`** matches `feynman_rules`.

Operators are configured with **`lagrangian.operator_action`** (small helpers like `replacement_operator`); everything else below uses the **model API** on the compiled object.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from symbolica import Expression, S

from model import Model, PartialD, dirac_field, flavor_index, scalar_field
from lagrangian.operator_action import FieldOperator, replacement_operator, single_field_result

## 1. Define fields and build the model

In [ ]:
mu = S("mu")

Phi = scalar_field("Phi", self_conjugate=True)
Chi = scalar_field("Chi", self_conjugate=True)

model = Model(
    fields=(Phi, Chi),
    lagrangian_decl=Phi * Phi * Phi + Phi * PartialD(Phi, mu),
)

L = model.lagrangian()
print("Source (first declared piece):", model.lagrangian_decl.source_terms[0])
print("Compiled terms:", len(L.terms))

## 2. Inspect compiled terms (ordered metadata)

Each term has `fields`, `derivatives`, `coupling`, and optional `closed_dirac_bilinears`. This is the authoritative ordering for fermions/ghosts.

In [ ]:
def field_factor(occ):
    name = occ.field.name
    if occ.conjugated and not occ.field.self_conjugate:
        name = name + ".bar"
    return name


def describe_term(t, index):
    factors = " · ".join(field_factor(o) for o in t.fields)
    if t.derivatives:
        d = ", ".join(f"slot {a.target} → ∂_{a.lorentz_index}" for a in t.derivatives)
    else:
        d = "(none)"
    bil = t.closed_dirac_bilinears or "(none)"
    print(f"[{index}] {factors}")
    print(f"     ∂: {d}   bilinears: {bil}")


for i, term in enumerate(L.terms):
    describe_term(term, i)

## 3. Symbolica export (`to_symbolica`)

Use this for **printing, simplifying coefficients, or quick checks**. Do not treat the product order in Symbolica as physical fermion order—that information stays in `L.terms`.

In [ ]:
expr = L.to_symbolica()
print(expr)
print("Simplified:", Expression.parse(str(expr)).expand())

## 4. Apply a field operator (`apply_operator`)

**Simple replacement:** map every `Phi` factor to `Chi` (graded Leibniz over the product).

**Product replacement + derivative:** if one slot becomes several fields and the slot carried `∂`, the engine **fans out** derivatives across the new slots (bosonic Leibniz), so gauge-like `φ → χ φ` on `φ ∂φ` is consistent.

In [ ]:
replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
L_replaced = L.apply_operator(replace_phi)

print("After replacing each Phi by Chi:", len(L_replaced.terms), "term(s)")
for i, term in enumerate(L_replaced.terms):
    describe_term(term, i)

print("\nSymbolica view:")
print(L_replaced.to_symbolica())

In [ ]:
split_phi = replacement_operator(
    "split_phi",
    {Phi: single_field_result((Chi(), Phi()))},
)
L_split = L.apply_operator(split_phi)

print("Product replacement on ∂ slots →", len(L_split.terms), "terms (derivative Leibniz).")
for i, term in enumerate(L_split.terms):
    describe_term(term, i)

## 5. Flavor-expanded export (`flavor_expand`)

Same keyword as for vertices: pass **`flavor_expand=True`** (or specific flavor indices) so **`to_symbolica`** serializes the **expanded** interaction list (class members like `e`, `mu`, `ta` instead of the generic `l`).

You can pass the same flag to **`apply_operator(..., flavor_expand=True)`** if you want the operator to act on that expanded view first.

In [ ]:
Generation = flavor_index("Generation", 3, prefix="f")
lepton = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
)
H = scalar_field("H", self_conjugate=True)
f = S("f")

flavor_model = Model(
    fields=(lepton, H),
    lagrangian_decl=S("g") * lepton.bar(f) * lepton(f) * H,
)
L_fl = flavor_model.lagrangian()

compact = L_fl.to_symbolica()
expanded = L_fl.to_symbolica(flavor_expand=True)

print("Generic (class field 'l'):")
print(compact)
print()
print("Flavor-expanded (e, mu, ta):")
print(expanded)

## 6. Optional: odd operator sign (fermion bookkeeping)

For **odd** operators (`parity=1`), the graded Leibniz rule inserts a **minus** when the operator crosses an odd number of fermionic factors to the left.

Compiled `psibar * psi` terms carry **`closed_dirac_bilinears`** metadata. If you replace a fermion inside that structure, the replacement must still expose exactly one matching **conjugated** (`psibar`) or **unconjugated** (`psi`) Dirac factor—here we map `psi → eta` while preserving that conjugation flag.

In [ ]:
psi = dirac_field("psi", indices=())
eta = dirac_field("eta", indices=())

fermion_model = Model(
    fields=(psi, eta),
    lagrangian_decl=psi.bar() * psi(),
)
L_f = fermion_model.lagrangian()


def odd_s_on_psi(occurrence):
    if occurrence.field is not psi:
        return None
    # Preserve .bar vs plain so closed_dirac_bilinears stay valid.
    return single_field_result(eta.occurrence(conjugated=occurrence.conjugated))


s_odd = FieldOperator(name="s", parity=1, on_field=odd_s_on_psi)
L_s = L_f.apply_operator(s_odd)

print("Original:", len(L_f.terms), "term(s)")
for t in L_f.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

print("\nAfter odd s on each psi slot:", len(L_s.terms), "terms")
for t in L_s.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))